# Analisando dados das eleições no Brasil

Dados do TSE — resultados eleitorais: https://dadosabertos.tse.jus.br/.
No site do TSE dá pra fazer consultas bem específicas: https://sig.tse.jus.br/ords/dwapr/r/seai/sig-eleicao-resultados/resultado-da-elei%C3%A7%C3%A3o?p0_abrangencia=Munic%C3%ADpio&clear=RP&session=13649455056979

1998 o arquivo está errado

Objetivo: ver quais municípios “acertam” o resultado nacional em cada eleição.

**Versão tse2 (critério mais rígido):** no 1º turno de cada ano exige acertar o **1º colocado** (ACERTO1) e o **2º colocado** (ACERTO1b) a nível nacional; o 2º turno continua só com o vencedor (ACERTO2).

## Como rodar outro ano

1. Na **célula de configuração**, altere `ANO` (e, se precisar, o nome do CSV original do TSE).
2. Execute as células **na ordem**, uma a uma, até gerar `votos_mun_{ANO}.csv` e `acertos_mun_{ANO}.csv`.
3. Repita para o próximo ano (ex.: 2022 → 2018 → 2014 → 2010 → 2006 → 2002 → 1998).

Arquivo original esperado (padrão TSE): `votacao_secao_{ANO}_BR.csv` (votação por seção, presidente, Brasil).

Fluxo para cada ano (exemplo para 2018):
- Baixar o CSV do TSE na pasta `estatisticas/tse/`
- Na **config**, `ANO` e (se preciso) `NOME_ARQUIVO_ORIGINAL`
- Rodar **config** → detectar tipo → **passo 0 ou 0b** → **passo 1** (total nacional) → **passo 2** → **passo 3**
- Repetir para outro ano; no final rode o **cruzamento multi-ano**

**Dois formatos de arquivo** (a config detecta sozinha):
| Tipo | Exemplo | Granularidade |
|------|---------|---------------|
| `secao` | `votacao_secao_2002_BR.csv` | seção eleitoral (2002–2022) |
| `munzona` | `votacao_candidato_munzona_1998_BRASIL.csv` | município + zona (anos antigos) |

In [22]:
# Configuração — altere aqui e rode esta célula antes das demais
import pandas as pd
from pathlib import Path

# === ALTERE PARA O ANO QUE VOCÊ ESTÁ PROCESSANDO ===
ANO = 1994
# Nome do CSV baixado do TSE (None = tenta secao, depois munzona)
NOME_ARQUIVO_ORIGINAL = None
# NOME_ARQUIVO_ORIGINAL = "votacao_candidato_munzona_1998_BRASIL.csv"

PASTA = Path(".")


def candidatos_arquivo_original(ano):
    return [
        PASTA / f"votacao_secao_{ano}_BR.csv",
        PASTA / f"votacao_candidato_munzona_{ano}_BRASIL.csv",
        PASTA / f"votacao_candidato_munzona_{ano}_BR.csv",
    ]


def resolver_arquivo_original(ano, nome_manual=None):
    if nome_manual:
        return PASTA / nome_manual
    for caminho in candidatos_arquivo_original(ano):
        if caminho.exists():
            return caminho
    return candidatos_arquivo_original(ano)[0]


ARQUIVO_ORIGINAL = resolver_arquivo_original(ANO, NOME_ARQUIVO_ORIGINAL)
ARQUIVO = PASTA / f"votos_{ANO}.csv"
ARQUIVO_MUN = PASTA / f"votos_mun_{ANO}.csv"
ARQUIVO_ACERTOS = PASTA / f"acertos_mun_{ANO}.csv"

ENCODING = "latin-1"
SEPARADOR = ";"
VOTOS_INVALIDOS = {"95", "96", "97"}

COLUNAS_CHAVE = [
    "NR_TURNO",
    "SG_UF",
    "CD_MUNICIPIO",
    "NM_MUNICIPIO",
    "NR_ZONA",
    "NR_SECAO",
    "NR_VOTAVEL",
    "NM_VOTAVEL",
    "QT_VOTOS",
]

# Chave entre anos: só CD_MUNICIPIO (nome da cidade pode mudar, ex. SANTO ANDRE / SANTO ANDRÉ)
CHAVE_CRUZAMENTO = "CD_MUNICIPIO"
CHAVES_MUN = ["NR_TURNO", "CD_MUNICIPIO", "NR_VOTAVEL", "NM_VOTAVEL"]
CHAVES_CIDADE = [CHAVE_CRUZAMENTO]

# UF e nome entram nos CSV finais, mas não no merge entre anos
COLUNAS_MUN_EXTRA = ["SG_UF", "NM_MUNICIPIO"]
COLUNAS_ACERTOS_SAIDA = [
    "SG_UF",
    "CD_MUNICIPIO",
    "NM_MUNICIPIO",
    "ACERTO1",
    "ACERTO1b",
    "ACERTO2",
]


def referencia_municipios(df):
    """Uma linha por município com UF e nome (do ano que está sendo processado)."""
    return (
        df[["CD_MUNICIPIO", "SG_UF", "NM_MUNICIPIO"]]
        .drop_duplicates(subset=["CD_MUNICIPIO"], keep="first")
    )


def detectar_tipo_arquivo(arquivo):
    if not arquivo.exists():
        return None
    cols = set(pd.read_csv(arquivo, sep=SEPARADOR, encoding=ENCODING, nrows=0).columns)
    if "NR_VOTAVEL" in cols:
        return "secao"
    if "NR_CANDIDATO" in cols and "QT_VOTOS_NOMINAIS_VALIDOS" in cols:
        return "munzona"
    raise ValueError(f"Formato não reconhecido em {arquivo.name}: {sorted(cols)[:12]}...")


def normalizar_cd_municipio(serie):
    return serie.astype(str).str.strip().str.lstrip("0").replace("", "0")


TIPO_ARQUIVO = detectar_tipo_arquivo(ARQUIVO_ORIGINAL) if ARQUIVO_ORIGINAL.exists() else None

print("--- Configuração ---")
print(f"Chave para cruzar anos: {CHAVE_CRUZAMENTO}")
print(f"Ano: {ANO}")
print(f"Original TSE: {ARQUIVO_ORIGINAL.name} {'(ok)' if ARQUIVO_ORIGINAL.exists() else '(não encontrado)'}")
print(f"Tipo detectado: {TIPO_ARQUIVO or 'arquivo ausente'}")
print(f"Seções:       {ARQUIVO.name}  (só tipo secao)")
print(f"Municípios:   {ARQUIVO_MUN.name}")
print(f"Acertos:      {ARQUIVO_ACERTOS.name}")

--- Configuração ---
Chave para cruzar anos: CD_MUNICIPIO
Ano: 1994
Original TSE: votacao_secao_1994_BR.csv (não encontrado)
Tipo detectado: arquivo ausente
Seções:       votos_1994.csv  (só tipo secao)
Municípios:   votos_mun_1994.csv
Acertos:      acertos_mun_1994.csv


## Adaptadores — anos com arquivo `munzona` (ex.: 1998)

Diferenças principais em relação a `votacao_secao_*` (ver `leiame.pdf`, tabela **VOTACAO_CANDIDATO_MUNZONA**):

| Secao (2002+) | Munzona (1998 etc.) |
|---------------|---------------------|
| `NR_VOTAVEL`, `NM_VOTAVEL` | `NR_CANDIDATO`, `NM_URNA_CANDIDATO` |
| `QT_VOTOS` | `QT_VOTOS_NOMINAIS_VALIDOS` (já só votos válidos do candidato) |
| 1 linha por **seção** | 1 linha por **município + zona** (somamos zonas → município) |
| Branco/nulo como votável 95/96/97 | Não vêm nesse arquivo |
| `CD_MUNICIPIO` sem zeros à esquerda | Pode vir `01120` → normalizamos para `1120` |

**1998:** FHC venceu no **1º turno**; no arquivo não há 2º turno para presidente (`ACERTO2` fica vazio nesse ano).

In [18]:
# Funções para tipo munzona + verificação antes de processar
COLUNAS_MUNZONA = [
    "NR_TURNO",
    "SG_UF",
    "CD_MUNICIPIO",
    "NM_MUNICIPIO",
    "CD_CARGO",
    "NR_CANDIDATO",
    "NM_URNA_CANDIDATO",
    "QT_VOTOS_NOMINAIS_VALIDOS",
    "ST_VOTO_EM_TRANSITO",
]


def verificar_arquivo_ano(arquivo=ARQUIVO_ORIGINAL, ano=ANO):
    """Resumo do que o pipeline vai usar neste ano."""
    if not arquivo.exists():
        print(f"Arquivo não encontrado: {arquivo}")
        return None
    tipo = detectar_tipo_arquivo(arquivo)
    print(f"Ano {ano} | {arquivo.name} | tipo: {tipo}")
    if tipo == "secao":
        print("  → Use passo 0 (seções) → passo 2 → passo 3")
        return tipo
    if tipo != "munzona":
        return tipo
    CD_CARGO_PRESIDENTE = "1"
    turnos_pres = set()
    cargos = {}
    tem_pres_t2 = False
    for chunk in pd.read_csv(
        arquivo, sep=SEPARADOR, encoding=ENCODING, usecols=["NR_TURNO", "CD_CARGO", "DS_CARGO"], dtype=str, chunksize=400_000
    ):
        for c, d in chunk[["CD_CARGO", "DS_CARGO"]].drop_duplicates().values:
            cargos[c] = d
        pres = chunk[chunk["CD_CARGO"] == CD_CARGO_PRESIDENTE]
        if pres.empty:
            continue
        turnos_pres.update(pres["NR_TURNO"].unique())
        if (pres["NR_TURNO"] == "2").any():
            tem_pres_t2 = True
    print("  → Use passo 0b (munzona direto em votos_mun) → passo 3")
    print(f"  Cargos no arquivo (todos): {dict(sorted(cargos.items(), key=lambda x: x[0]))}")
    print(f"  Filtro na geração: CD_CARGO == {CD_CARGO_PRESIDENTE} e ST_VOTO_EM_TRANSITO == N")
    print(f"  Turnos do presidente: {sorted(turnos_pres)}")
    print(f"  Presidente com 2º turno: {'sim' if tem_pres_t2 else 'não (só ACERTO1 faz sentido)'}")
    return tipo


def gerar_votos_mun_munzona(
    arquivo_entrada=ARQUIVO_ORIGINAL,
    arquivo_saida=ARQUIVO_MUN,
    forcar=False,
):
    if not arquivo_entrada.exists():
        raise FileNotFoundError(arquivo_entrada)
    precisa = (
        forcar
        or not arquivo_saida.exists()
        or arquivo_saida.stat().st_mtime < arquivo_entrada.stat().st_mtime
    )
    if not precisa:
        print(f"Arquivo já existe: {arquivo_saida.name}")
        return pd.read_csv(arquivo_saida, sep=SEPARADOR, encoding=ENCODING, dtype=str)

    print(f"Gerando {arquivo_saida.name} (munzona → município) ...")
    partes = []
    for chunk in pd.read_csv(
        arquivo_entrada,
        sep=SEPARADOR,
        encoding=ENCODING,
        usecols=COLUNAS_MUNZONA,
        dtype=str,
        chunksize=500_000,
    ):
        # Só presidente (CD_CARGO=1); o arquivo munzona traz outros cargos também
        filtro = chunk[
            (chunk["CD_CARGO"] == "1")
            & (chunk["ST_VOTO_EM_TRANSITO"] == "N")
        ].copy()
        filtro[CHAVE_CRUZAMENTO] = normalizar_cd_municipio(filtro["CD_MUNICIPIO"])
        filtro["QT_VOTOS"] = pd.to_numeric(filtro["QT_VOTOS_NOMINAIS_VALIDOS"], errors="coerce").fillna(0).astype(int)
        filtro = filtro.rename(
            columns={"NR_CANDIDATO": "NR_VOTAVEL", "NM_URNA_CANDIDATO": "NM_VOTAVEL"}
        )
        partes.append(
            filtro.groupby(CHAVES_MUN, as_index=False).agg(
                QT_VOTOS=("QT_VOTOS", "sum"),
                SG_UF=("SG_UF", "first"),
                NM_MUNICIPIO=("NM_MUNICIPIO", "first"),
            )
        )
    votos_mun = (
        pd.concat(partes, ignore_index=True)
        .groupby(CHAVES_MUN, as_index=False)
        .agg(QT_VOTOS=("QT_VOTOS", "sum"), SG_UF=("SG_UF", "first"), NM_MUNICIPIO=("NM_MUNICIPIO", "first"))
    )
    votos_mun[CHAVE_CRUZAMENTO] = normalizar_cd_municipio(votos_mun[CHAVE_CRUZAMENTO])
    votos_mun = votos_mun[
        ["NR_TURNO", CHAVE_CRUZAMENTO, "SG_UF", "NM_MUNICIPIO", "NR_VOTAVEL", "NM_VOTAVEL", "QT_VOTOS"]
    ]
    votos_mun.to_csv(arquivo_saida, sep=SEPARADOR, encoding=ENCODING, index=False)
    print(f"Linhas: {len(votos_mun):,} | Turnos: {sorted(votos_mun['NR_TURNO'].unique())}")
    return votos_mun


if ARQUIVO_ORIGINAL.exists():
    verificar_arquivo_ano()
else:
    print("Coloque o CSV do TSE na pasta e rode a config de novo.")

Ano 1994 | votacao_candidato_munzona_1994_BRASIL.csv | tipo: munzona
  → Use passo 0b (munzona direto em votos_mun) → passo 3
  Cargos no arquivo (todos): {'1': 'Presidente', '3': 'Governador', '5': 'Senador', '6': 'Deputado Federal', '7': 'Deputado Estadual'}
  Filtro na geração: CD_CARGO == 1 e ST_VOTO_EM_TRANSITO == N
  Turnos do presidente: ['1']
  Presidente com 2º turno: não (só ACERTO1 faz sentido)


In [19]:
# Passo 0: gerar votos_{ANO}.csv (9 colunas) a partir do original do TSE
def gerar_arquivo_reduzido(forcar=False):
    if not ARQUIVO_ORIGINAL.exists():
        raise FileNotFoundError(f"Não encontrei: {ARQUIVO_ORIGINAL.resolve()}")

    precisa_gerar = (
        forcar
        or not ARQUIVO.exists()
        or ARQUIVO.stat().st_mtime < ARQUIVO_ORIGINAL.stat().st_mtime
    )
    if not precisa_gerar:
        print(f"Arquivo já existe: {ARQUIVO.name}")
        return

    print(f"Gerando {ARQUIVO.name} ...")
    primeira_parte = True
    linhas = 0

    for chunk in pd.read_csv(
        ARQUIVO_ORIGINAL,
        sep=SEPARADOR,
        encoding=ENCODING,
        usecols=COLUNAS_CHAVE,
        dtype={"NR_VOTAVEL": str, "NR_TURNO": str},
        chunksize=500_000,
    ):
        chunk.to_csv(
            ARQUIVO,
            sep=SEPARADOR,
            encoding=ENCODING,
            index=False,
            mode="w" if primeira_parte else "a",
            header=primeira_parte,
        )
        linhas += len(chunk)
        primeira_parte = False
        print(f"  {linhas:,} linhas gravadas", end="\r")

    print(f"\nPronto: {linhas:,} linhas")
    print(f"Original: {ARQUIVO_ORIGINAL.stat().st_size / 1e9:.2f} GB")
    print(f"Reduzido: {ARQUIVO.stat().st_size / 1e9:.2f} GB")


if TIPO_ARQUIVO == "secao":
    gerar_arquivo_reduzido()
    # gerar_arquivo_reduzido(forcar=True)
elif TIPO_ARQUIVO == "munzona":
    print("Tipo munzona: pule este passo e rode o passo 0b na próxima célula.")
else:
    print("Defina ANO/NOME_ARQUIVO_ORIGINAL e garanta que o CSV existe.")

Tipo munzona: pule este passo e rode o passo 0b na próxima célula.


In [20]:
# Passo 0b: munzona → votos_mun_{ANO}.csv (pule o passo 0 e o passo 2)
votos_mun = gerar_votos_mun_munzona()
# votos_mun = gerar_votos_mun_munzona(forcar=True)
display(votos_mun.head(8))

Gerando votos_mun_1994.csv (munzona → município) ...
Linhas: 40,119 | Turnos: ['1']


,NR_TURNO,CD_MUNICIPIO,SG_UF,NM_MUNICIPIO,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS
0,1,10006,PI,BRASILEIRA,11,ESPERIDIAO AMIN HELOU FILHO,210
1,1,10006,PI,BRASILEIRA,12,LEONEL DE MOURA BRIZOLA,32
2,1,10006,PI,BRASILEIRA,13,LUIZ INACIO LULA DA SILVA,411
3,1,10006,PI,BRASILEIRA,15,ORESTES QUERCIA,402
4,1,10006,PI,BRASILEIRA,20,HERNANI GOULART FORTUNA,11
5,1,10006,PI,BRASILEIRA,36,CARLOS ANTONIO GOMES,81
6,1,10006,PI,BRASILEIRA,45,FERNANDO HENRIQUE CARDOSO,1806
7,1,10006,PI,BRASILEIRA,56,ENEAS FERREIRA CARNEIRO,26


In [21]:
# Passo 1: resultado nacional (conferir com o TSE) — funciona em secao e munzona
def somar_votos_secao(arquivo, turno):
    totais = {}
    for chunk in pd.read_csv(
        arquivo,
        sep=SEPARADOR,
        encoding=ENCODING,
        usecols=["NR_TURNO", "NR_VOTAVEL", "NM_VOTAVEL", "QT_VOTOS"],
        dtype={"NR_VOTAVEL": str, "NR_TURNO": str},
        chunksize=500_000,
    ):
        filtro = chunk[chunk["NR_TURNO"] == turno]
        agg = filtro.groupby(["NR_VOTAVEL", "NM_VOTAVEL"], as_index=False)["QT_VOTOS"].sum()
        for _, row in agg.iterrows():
            chave = (row["NR_VOTAVEL"], row["NM_VOTAVEL"])
            totais[chave] = totais.get(chave, 0) + int(row["QT_VOTOS"])
    return (
        pd.DataFrame(
            [(k[0], k[1], v) for k, v in totais.items()],
            columns=["NR_VOTAVEL", "NM_VOTAVEL", "QT_VOTOS"],
        )
        .sort_values("QT_VOTOS", ascending=False)
        .reset_index(drop=True)
    )


def somar_votos_votos_mun(arquivo_mun, turno):
    df = pd.read_csv(arquivo_mun, sep=SEPARADOR, encoding=ENCODING, dtype=str)
    df["QT_VOTOS"] = pd.to_numeric(df["QT_VOTOS"], errors="coerce").fillna(0).astype(int)
    t = df[df["NR_TURNO"] == turno]
    if t.empty:
        return pd.DataFrame(columns=["NR_VOTAVEL", "NM_VOTAVEL", "QT_VOTOS"])
    return (
        t.groupby(["NR_VOTAVEL", "NM_VOTAVEL"], as_index=False)["QT_VOTOS"]
        .sum()
        .sort_values("QT_VOTOS", ascending=False)
        .reset_index(drop=True)
    )


def exibir_resultado_nacional(turno):
    if ARQUIVO.exists() and (TIPO_ARQUIVO == "secao" or detectar_tipo_arquivo(ARQUIVO) == "secao"):
        print(f"Fonte: {ARQUIVO.name} (por seção)")
        resultado = somar_votos_secao(ARQUIVO, turno)
        tem_branco_nulo = True
    elif ARQUIVO_MUN.exists():
        print(f"Fonte: {ARQUIVO_MUN.name} (soma dos municípios; munzona não traz branco/nulo)")
        resultado = somar_votos_votos_mun(ARQUIVO_MUN, turno)
        tem_branco_nulo = False
    else:
        print(
            "Nenhuma fonte pronta. Rode o passo 0 (secao) ou 0b (munzona) antes, "
            f"ou confira se existe {ARQUIVO.name} ou {ARQUIVO_MUN.name}."
        )
        return None

    if resultado.empty:
        print(f"  Turno {turno}: sem dados de presidente neste ano.")
        return resultado

    validos = resultado[~resultado["NR_VOTAVEL"].isin(VOTOS_INVALIDOS)].copy()
    validos["pct_validos"] = (validos["QT_VOTOS"] / validos["QT_VOTOS"].sum() * 100).round(2)

    print(f"\n--- {turno}º turno — Brasil (votos válidos, %) ---")
    display(validos)

    if tem_branco_nulo:
        invalidos = resultado[resultado["NR_VOTAVEL"].isin(VOTOS_INVALIDOS)]
        if not invalidos.empty:
            print("Brancos, nulos e anulados:")
            display(invalidos)

    vencedor = validos.iloc[0]
    print(
        f"Vencedor no arquivo: {vencedor['NM_VOTAVEL']} "
        f"(nº {vencedor['NR_VOTAVEL']}) — {vencedor['QT_VOTOS']:,} votos válidos "
        f"({vencedor['pct_validos']}%)"
    )
    return resultado


print(f"Ano {ANO} | tipo {TIPO_ARQUIVO}")
for turno in ["1", "2"]:
    exibir_resultado_nacional(turno)

# Referência oficial 1998 (1º turno, TSE — eleição definida no 1º turno)
if ANO == 1998:
    print("\nReferência TSE 1998 (1º turno, votos válidos — site/Wikipedia):")
    print("  FHC (45): ~35.936.382 | Lula (13): ~21.475.211 | Ciro (23): ~7.426.187")


Ano 1994 | tipo munzona
Fonte: votos_mun_1994.csv (soma dos municípios; munzona não traz branco/nulo)

--- 1º turno — Brasil (votos válidos, %) ---


,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS,pct_validos
0,45,FERNANDO HENRIQUE CARDOSO,34350217,54.28
1,13,LUIZ INACIO LULA DA SILVA,17112255,27.04
2,56,ENEAS FERREIRA CARNEIRO,4670894,7.38
3,15,ORESTES QUERCIA,2771788,4.38
4,12,LEONEL DE MOURA BRIZOLA,2015284,3.18
5,11,ESPERIDIAO AMIN HELOU FILHO,1739458,2.75
6,36,CARLOS ANTONIO GOMES,387611,0.61
7,20,HERNANI GOULART FORTUNA,238126,0.38


Vencedor no arquivo: FERNANDO HENRIQUE CARDOSO (nº 45) — 34,350,217 votos válidos (54.28%)
Fonte: votos_mun_1994.csv (soma dos municípios; munzona não traz branco/nulo)
  Turno 2: sem dados de presidente neste ano.


## Colunas do arquivo do TSE (`votacao_secao_{ANO}_BR.csv`)

**Granularidade:** cada linha = quantidade de votos (`QT_VOTOS`) que um **votável** recebeu em uma **seção** (`NR_ZONA` + `NR_SECAO`) de um **município**.

### Onde está a cidade?
| Coluna | Uso |
|--------|-----|
| `CD_MUNICIPIO` | Código TSE do município (chave única — use nos cruzamentos) |
| `NM_MUNICIPIO` / `SG_UF` | Referência humana; o nome pode mudar entre anos |

`SG_UE` / `NM_UE` valem "BR"/"BRASIL" para presidente — **não** identificam a cidade.

### Votos válidos vs inválidos (`NR_VOTAVEL`)
| Código | Significado | Contagem |
|--------|-------------|----------|
| Número do candidato (ex. 13, 22) | Voto nominal | **Válido** |
| 95 | Voto em branco | Inválido |
| 96 | Voto nulo | Inválido |
| 97 | Voto anulado (apurado em separado) | Inválido |

**Votos válidos** = soma de `QT_VOTOS` onde `NR_VOTAVEL` é candidato (excluir 95, 96, 97).

**Resultado no município ou no país** = candidato com mais votos válidos no turno; percentual = votos do candidato ÷ total de votos válidos.

### Filtro útil
- `NR_TURNO == "1"` ou `"2"` → primeiro ou segundo turno

### Colunas que você pode ignorar no início
Metadados de extração (`DT_GERACAO`, `HH_GERACAO`), tipo de eleição, local de votação (`NR_LOCAL_VOTACAO`, endereço), `SQ_CANDIDATO` (só para cruzar com outro arquivo do TSE).

### Arquivos gerados por ano
- `votos_{ANO}.csv` — seções (passo 0)
- `votos_mun_{ANO}.csv` — totais por município e candidato (com UF e nome)
- `acertos_mun_{ANO}.csv` — 1 linha por município: ACERTO1 (1º colocado no 1º turno), ACERTO1b (2º colocado no 1º turno), ACERTO2 (2º turno)

**Cruzamento entre anos:** use apenas `CD_MUNICIPIO`. O nome do município pode variar (ex.: `SANTO ANDRE` em 2002 e `SANTO ANDRÉ` depois).

In [22]:
# (Opcional) Amostra do arquivo de seções — só existe após o passo 0 em anos "secao"
if ARQUIVO.exists():
    amostra = pd.read_csv(ARQUIVO, sep=SEPARADOR, encoding=ENCODING, nrows=10_000, dtype=str)
    print(f"Amostra de {ARQUIVO.name}:")
    display(amostra.head(5))
else:
    print(f"{ARQUIVO.name} não existe (normal em anos munzona). Use o passo 1 acima para o total nacional.")

votos_1994.csv não existe (normal em anos munzona). Use o passo 1 acima para o total nacional.


In [23]:
# (Opcional) Votação por município — exemplo Santo André (CD 70572)
# Precisa de votos_{ANO}.csv (passo 0) ou votos_mun_{ANO}.csv (passo 0b).
def votos_por_municipio(arquivo, turno="2"):
    chaves = ["CD_MUNICIPIO", "NR_VOTAVEL", "NM_VOTAVEL"]
    partes = []
    for chunk in pd.read_csv(
        arquivo,
        sep=SEPARADOR,
        encoding=ENCODING,
        usecols=["NR_TURNO", "CD_MUNICIPIO", "NR_VOTAVEL", "NM_VOTAVEL", "QT_VOTOS"],
        dtype={"NR_VOTAVEL": str, "NR_TURNO": str, "CD_MUNICIPIO": str},
        chunksize=500_000,
    ):
        filtro = chunk[
            (chunk["NR_TURNO"] == turno)
            & (~chunk["NR_VOTAVEL"].isin(VOTOS_INVALIDOS))
        ]
        partes.append(filtro.groupby(chaves, as_index=False)["QT_VOTOS"].sum())
    return pd.concat(partes, ignore_index=True).groupby(chaves, as_index=False)["QT_VOTOS"].sum()


def votos_por_municipio_de_votos_mun(arquivo_mun, turno="2"):
    df = pd.read_csv(arquivo_mun, sep=SEPARADOR, encoding=ENCODING, dtype=str)
    df["QT_VOTOS"] = pd.to_numeric(df["QT_VOTOS"], errors="coerce")
    filtro = df[
        (df["NR_TURNO"] == turno) & (~df["NR_VOTAVEL"].isin(VOTOS_INVALIDOS))
    ]
    chaves = ["CD_MUNICIPIO", "NR_VOTAVEL", "NM_VOTAVEL"]
    return filtro.groupby(chaves, as_index=False)["QT_VOTOS"].sum()


CD_SANTO_ANDRE = "70572"
TURNO_EXEMPLO = "2"

if not ARQUIVO.exists() and not ARQUIVO_MUN.exists():
    print(
        f"Sem {ARQUIVO.name} nem {ARQUIVO_MUN.name}. "
        "Rode o passo 0 (anos secao) ou 0b (munzona) antes desta amostra."
    )
else:
    por_mun = None
    ref_ano = None
    if ARQUIVO.exists():
        por_mun = votos_por_municipio(ARQUIVO, turno=TURNO_EXEMPLO)
        if ARQUIVO_MUN.exists():
            ref_ano = referencia_municipios(
                pd.read_csv(ARQUIVO_MUN, sep=SEPARADOR, encoding=ENCODING, dtype=str)
            )
        else:
            ref_partes = []
            for chunk in pd.read_csv(
                ARQUIVO,
                sep=SEPARADOR,
                encoding=ENCODING,
                usecols=["CD_MUNICIPIO", "SG_UF", "NM_MUNICIPIO"],
                dtype=str,
                chunksize=500_000,
            ):
                ref_partes.append(chunk.drop_duplicates(subset=["CD_MUNICIPIO"]))
            ref_ano = pd.concat(ref_partes).drop_duplicates(subset=["CD_MUNICIPIO"])
    elif ARQUIVO_MUN.exists():
        turnos = sorted(
            pd.read_csv(
                ARQUIVO_MUN,
                sep=SEPARADOR,
                encoding=ENCODING,
                usecols=["NR_TURNO"],
                dtype=str,
            )["NR_TURNO"]
            .dropna()
            .unique()
        )
        turno_usar = TURNO_EXEMPLO
        if turno_usar not in turnos:
            if "1" in turnos:
                print(
                    f"Aviso: {ANO} não tem {turno_usar}º turno presidencial "
                    f"(turnos: {turnos}). Mostrando 1º turno."
                )
                turno_usar = "1"
            else:
                print(f"Turno {turno_usar} não existe em {ARQUIVO_MUN.name}. Turnos: {turnos}.")
                turno_usar = None
        if turno_usar:
            por_mun = votos_por_municipio_de_votos_mun(ARQUIVO_MUN, turno=turno_usar)
            ref_ano = referencia_municipios(
                pd.read_csv(ARQUIVO_MUN, sep=SEPARADOR, encoding=ENCODING, dtype=str)
            )

    if por_mun is not None:
        por_mun = ref_ano.merge(por_mun, on="CD_MUNICIPIO", how="right")
        stAndre = por_mun[por_mun["CD_MUNICIPIO"] == CD_SANTO_ANDRE].copy()
        if stAndre.empty:
            print(f"Nenhum voto para CD_MUNICIPIO={CD_SANTO_ANDRE} neste turno.")
        else:
            stAndre["pct_no_municipio"] = (
                stAndre["QT_VOTOS"] / stAndre["QT_VOTOS"].sum() * 100
            ).round(2)
            display(stAndre.sort_values("QT_VOTOS", ascending=False))

Aviso: 1994 não tem 2º turno presidencial (turnos: ['1']). Mostrando 1º turno.


,CD_MUNICIPIO,SG_UF,NM_MUNICIPIO,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS,pct_no_municipio
25377,70572,SP,SANTO ANDRE,13,LUIZ INACIO LULA DA SILVA,162136,46.23
25381,70572,SP,SANTO ANDRE,45,FERNANDO HENRIQUE CARDOSO,135555,38.65
25382,70572,SP,SANTO ANDRE,56,ENEAS FERREIRA CARNEIRO,33881,9.66
25378,70572,SP,SANTO ANDRE,15,ORESTES QUERCIA,10461,2.98
25375,70572,SP,SANTO ANDRE,11,ESPERIDIAO AMIN HELOU FILHO,6082,1.73
25376,70572,SP,SANTO ANDRE,12,LEONEL DE MOURA BRIZOLA,964,0.27
25379,70572,SP,SANTO ANDRE,20,HERNANI GOULART FORTUNA,854,0.24
25380,70572,SP,SANTO ANDRE,36,CARLOS ANTONIO GOMES,757,0.22


In [24]:
# Passo 2: totais por município e candidato → votos_mun_{ANO}.csv
def votos_por_municipio_todos(arquivo, turno=None):
    """Agrega votos válidos por CD_MUNICIPIO; turno=None traz 1º e 2º turnos."""
    usecols = CHAVES_MUN + ["QT_VOTOS"] + COLUNAS_MUN_EXTRA
    partes = []
    for chunk in pd.read_csv(
        arquivo,
        sep=SEPARADOR,
        encoding=ENCODING,
        usecols=usecols,
        dtype={"NR_VOTAVEL": str, "NR_TURNO": str, "CD_MUNICIPIO": str},
        chunksize=500_000,
    ):
        filtro = chunk[~chunk["NR_VOTAVEL"].isin(VOTOS_INVALIDOS)].copy()
        filtro[CHAVE_CRUZAMENTO] = normalizar_cd_municipio(filtro[CHAVE_CRUZAMENTO])
        if turno is not None:
            filtro = filtro[filtro["NR_TURNO"] == turno]
        partes.append(
            filtro.groupby(CHAVES_MUN, as_index=False).agg(
                QT_VOTOS=("QT_VOTOS", "sum"),
                SG_UF=("SG_UF", "first"),
                NM_MUNICIPIO=("NM_MUNICIPIO", "first"),
            )
        )
    agregado = pd.concat(partes, ignore_index=True).groupby(CHAVES_MUN, as_index=False).agg(
        QT_VOTOS=("QT_VOTOS", "sum"),
        SG_UF=("SG_UF", "first"),
        NM_MUNICIPIO=("NM_MUNICIPIO", "first"),
    )
    return agregado[
        ["NR_TURNO", "CD_MUNICIPIO", "SG_UF", "NM_MUNICIPIO", "NR_VOTAVEL", "NM_VOTAVEL", "QT_VOTOS"]
    ]


def gerar_votos_municipio(
    arquivo_entrada=ARQUIVO,
    arquivo_saida=ARQUIVO_MUN,
    forcar=False,
):
    precisa_gerar = (
        forcar
        or not arquivo_saida.exists()
        or (
            arquivo_entrada.exists()
            and arquivo_saida.stat().st_mtime < arquivo_entrada.stat().st_mtime
        )
    )
    if not precisa_gerar:
        print(f"Arquivo já existe: {arquivo_saida.name}")
        return pd.read_csv(arquivo_saida, sep=SEPARADOR, encoding=ENCODING, dtype=str)

    print(f"Gerando {arquivo_saida.name} ...")
    votos_mun = votos_por_municipio_todos(arquivo_entrada)
    votos_mun["QT_VOTOS"] = votos_mun["QT_VOTOS"].astype(int)
    votos_mun.to_csv(arquivo_saida, sep=SEPARADOR, encoding=ENCODING, index=False)

    print(f"Linhas: {len(votos_mun):,}")
    print(f"Tamanho: {arquivo_saida.stat().st_size / 1e6:.1f} MB")
    print(f"Turnos: {sorted(votos_mun['NR_TURNO'].unique())}")
    return votos_mun


if TIPO_ARQUIVO == "secao":
    votos_mun = gerar_votos_municipio()
    # votos_mun = gerar_votos_municipio(forcar=True)
    display(votos_mun.head(10))
elif ARQUIVO_MUN.exists():
    print(f"Tipo {TIPO_ARQUIVO}: votos_mun já gerado no passo 0b — {ARQUIVO_MUN.name}")
    votos_mun = pd.read_csv(ARQUIVO_MUN, sep=SEPARADOR, encoding=ENCODING, dtype=str)
    display(votos_mun.head(10))
else:
    print("Rode o passo 0 (secao) ou 0b (munzona) antes.")

# conferência: 2º turno em Santo André (CD 70572)
stAndre_arquivo = votos_mun[
    (votos_mun["NR_TURNO"] == "2") & (votos_mun["CD_MUNICIPIO"] == CD_SANTO_ANDRE)
].sort_values("QT_VOTOS", ascending=False)
print("\nSanto André (2º turno) no arquivo agregado:")
display(stAndre_arquivo)

Tipo munzona: votos_mun já gerado no passo 0b — votos_mun_1994.csv


,NR_TURNO,CD_MUNICIPIO,SG_UF,NM_MUNICIPIO,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS
0,1,10006,PI,BRASILEIRA,11,ESPERIDIAO AMIN HELOU FILHO,210
1,1,10006,PI,BRASILEIRA,12,LEONEL DE MOURA BRIZOLA,32
2,1,10006,PI,BRASILEIRA,13,LUIZ INACIO LULA DA SILVA,411
3,1,10006,PI,BRASILEIRA,15,ORESTES QUERCIA,402
4,1,10006,PI,BRASILEIRA,20,HERNANI GOULART FORTUNA,11
5,1,10006,PI,BRASILEIRA,36,CARLOS ANTONIO GOMES,81
6,1,10006,PI,BRASILEIRA,45,FERNANDO HENRIQUE CARDOSO,1806
7,1,10006,PI,BRASILEIRA,56,ENEAS FERREIRA CARNEIRO,26
8,1,10014,PI,AGRICOLANDIA,11,ESPERIDIAO AMIN HELOU FILHO,20
9,1,10014,PI,AGRICOLANDIA,12,LEONEL DE MOURA BRIZOLA,16



Santo André (2º turno) no arquivo agregado:


,NR_TURNO,CD_MUNICIPIO,SG_UF,NM_MUNICIPIO,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS


In [23]:
# Passo 3: acertos por município → acertos_mun_{ANO}.csv
def vencedor_nacional(votos_mun, turno):
    turno_df = votos_mun[votos_mun["NR_TURNO"] == turno]
    if turno_df.empty:
        return None
    totais = turno_df.groupby("NR_VOTAVEL")["QT_VOTOS"].sum()
    return totais.idxmax() if not totais.empty else None


def segundo_colocado_nacional(votos_mun, turno):
    turno_df = votos_mun[votos_mun["NR_TURNO"] == turno]
    if turno_df.empty:
        return None
    totais = (
        turno_df.groupby("NR_VOTAVEL")["QT_VOTOS"]
        .sum()
        .sort_values(ascending=False)
    )
    return totais.index[1] if len(totais) >= 2 else None


def acertos_por_turno(votos_mun, turno):
    """Acerto = 1 só com vencedor único no município e igual ao nacional; empate local = 0."""
    turno_df = votos_mun[votos_mun["NR_TURNO"] == turno]
    venc_nacional = vencedor_nacional(votos_mun, turno)

    max_votos = turno_df.groupby(CHAVES_CIDADE)["QT_VOTOS"].transform("max")
    lideres = turno_df[turno_df["QT_VOTOS"] == max_votos]
    sem_empate = lideres.groupby(CHAVES_CIDADE).filter(lambda g: len(g) == 1)

    resultado = turno_df[CHAVES_CIDADE].drop_duplicates()
    marcacao = sem_empate[CHAVES_CIDADE].copy()
    marcacao["acerto"] = (sem_empate["NR_VOTAVEL"].values == venc_nacional).astype(int)

    resultado = resultado.merge(marcacao, on=CHAVES_CIDADE, how="left")
    resultado["acerto"] = resultado["acerto"].fillna(0).astype(int)  # empate → 0
    return resultado


def acertos_segundo_colocado(votos_mun, turno):
    """Acerto = 1 se o 2º colocado local (sem empate) = 2º colocado nacional; empate no 2º = 0."""
    turno_df = votos_mun[votos_mun["NR_TURNO"] == turno]
    segundo_nacional = segundo_colocado_nacional(votos_mun, turno)

    resultado = turno_df[CHAVES_CIDADE].drop_duplicates()
    if segundo_nacional is None:
        resultado["acerto"] = 0
        return resultado

    marcacoes = []
    for _, grp in turno_df.groupby(CHAVES_CIDADE, sort=False):
        g = grp.sort_values("QT_VOTOS", ascending=False)
        posicoes = g["QT_VOTOS"].unique()
        acerto = 0
        if len(posicoes) >= 2:
            votos_2 = posicoes[1]
            cand_2 = g[g["QT_VOTOS"] == votos_2]
            if len(cand_2) == 1:
                acerto = int(cand_2.iloc[0]["NR_VOTAVEL"] == segundo_nacional)
        marcacoes.append(
            {CHAVE_CRUZAMENTO: g.iloc[0][CHAVE_CRUZAMENTO], "acerto": acerto}
        )

    marcacao = pd.DataFrame(marcacoes)
    resultado = resultado.merge(marcacao, on=CHAVES_CIDADE, how="left")
    resultado["acerto"] = resultado["acerto"].fillna(0).astype(int)
    return resultado


def gerar_acertos_municipio(
    arquivo_entrada=ARQUIVO_MUN,
    arquivo_saida=ARQUIVO_ACERTOS,
    forcar=False,
):
    precisa_gerar = (
        forcar
        or not arquivo_saida.exists()
        or (
            arquivo_entrada.exists()
            and arquivo_saida.stat().st_mtime < arquivo_entrada.stat().st_mtime
        )
    )
    if not precisa_gerar:
        print(f"Arquivo já existe: {arquivo_saida.name}")
        return pd.read_csv(arquivo_saida, sep=SEPARADOR, encoding=ENCODING, dtype=str)

    votos = pd.read_csv(arquivo_entrada, sep=SEPARADOR, encoding=ENCODING, dtype=str)
    votos["QT_VOTOS"] = pd.to_numeric(votos["QT_VOTOS"])

    ref = referencia_municipios(votos)
    t1 = acertos_por_turno(votos, "1").rename(columns={"acerto": "ACERTO1"})
    t1b = acertos_segundo_colocado(votos, "1").rename(columns={"acerto": "ACERTO1b"})
    acertos = t1.merge(t1b, on=CHAVES_CIDADE, how="inner")
    tem_turno2 = (votos["NR_TURNO"] == "2").any()

    if tem_turno2:
        t2 = acertos_por_turno(votos, "2").rename(columns={"acerto": "ACERTO2"})
        acertos = acertos.merge(t2, on=CHAVES_CIDADE, how="inner")
    else:
        print("Sem 2º turno presidencial neste ano — ACERTO2 ficará em branco.")
        acertos["ACERTO2"] = ""

    acertos = ref.merge(acertos, on=CHAVE_CRUZAMENTO, how="right")[COLUNAS_ACERTOS_SAIDA]

    acertos.to_csv(arquivo_saida, sep=SEPARADOR, encoding=ENCODING, index=False)

    print("Vencedor nacional 1º turno:", vencedor_nacional(votos, "1"))
    print("2º colocado nacional 1º turno:", segundo_colocado_nacional(votos, "1"))
    if tem_turno2:
        print("Vencedor nacional 2º turno:", vencedor_nacional(votos, "2"))
    print(f"Linhas: {len(acertos):,}")
    print(f"Tamanho: {arquivo_saida.stat().st_size / 1e6:.2f} MB")

    def contar_empates(turno):
        d = votos[votos["NR_TURNO"] == turno]
        if d.empty:
            return 0
        max_v = d.groupby(CHAVES_CIDADE)["QT_VOTOS"].transform("max")
        return (d[d["QT_VOTOS"] == max_v].groupby(CHAVES_CIDADE).size() > 1).sum()

    def contar_empates_2o_colocado(turno):
        d = votos[votos["NR_TURNO"] == turno]
        if d.empty:
            return 0
        empates = 0
        for _, grp in d.groupby(CHAVES_CIDADE, sort=False):
            pos = grp["QT_VOTOS"].unique()
            if len(pos) >= 2 and (grp["QT_VOTOS"] == pos[1]).sum() > 1:
                empates += 1
        return empates

    print(f"Empates no 1º turno — 1º colocado (ACERTO1=0): {contar_empates('1'):,}")
    print(f"Empates no 1º turno — 2º colocado (ACERTO1b=0): {contar_empates_2o_colocado('1'):,}")
    print(f"Acertaram 1º colocado no 1º turno (ACERTO1): {acertos['ACERTO1'].astype(int).sum():,}")
    print(f"Acertaram 2º colocado no 1º turno (ACERTO1b): {acertos['ACERTO1b'].astype(int).sum():,}")
    print(
        f"Acertaram 1º e 2º colocados no 1º turno: "
        f"{(acertos['ACERTO1'].astype(int) & acertos['ACERTO1b'].astype(int)).sum():,}"
    )
    if tem_turno2:
        print(f"Empates no 2º turno (ACERTO2=0): {contar_empates('2'):,}")
        print(f"Acertaram 2º turno: {acertos['ACERTO2'].astype(int).sum():,}")
        print(
            f"Acertaram 1º+2º no 1º turno e vencedor no 2º: "
            f"{(acertos['ACERTO1'].astype(int) & acertos['ACERTO1b'].astype(int) & acertos['ACERTO2'].astype(int)).sum():,}"
        )
    return acertos


acertos_mun = gerar_acertos_municipio()
# acertos_mun = gerar_acertos_municipio(forcar=True)  # descomente para regerar
display(acertos_mun.head(10))
display(acertos_mun.sample(5, random_state=42))

Sem 2º turno presidencial neste ano — ACERTO2 ficará em branco.
Vencedor nacional 1º turno: 45
2º colocado nacional 1º turno: 13
Linhas: 5,019
Tamanho: 0.14 MB
Empates no 1º turno — 1º colocado (ACERTO1=0): 4
Empates no 1º turno — 2º colocado (ACERTO1b=0): 223
Acertaram 1º colocado no 1º turno (ACERTO1): 4,493
Acertaram 2º colocado no 1º turno (ACERTO1b): 4,028
Acertaram 1º e 2º colocados no 1º turno: 4,004


,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,ACERTO1,ACERTO1b,ACERTO2
0,PI,10006,BRASILEIRA,1,1,
1,PI,10014,AGRICOLANDIA,1,1,
2,PI,10022,ALEGRETE DO PIAUI,1,1,
3,PI,10030,AGUA BRANCA,1,1,
4,PI,10049,BAIXA GRANDE DO RIBEIRO,1,0,
5,PI,10057,ALTO LONGA,1,1,
6,PI,10065,BOM PRINCIPIO DO PIAUI,1,0,
7,AC,1007,BUJARI,1,1,
8,PI,10073,ALTOS,0,0,
9,PI,10081,BONFIM DO PIAUI,1,1,


,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,ACERTO1,ACERTO1b,ACERTO2
2600,RJ,59099,SAQUAREMA,1,1,
3000,SP,67156,MOJI GUACU,1,1,
4394,RS,88510,SANTIAGO,1,1,
2387,PA,5371,SANTAREM NOVO,1,1,
1187,BA,33456,ARATUIPE,1,1,


In [25]:
# Cruzamento entre anos: só por CD_MUNICIPIO; UF/nome vêm do arquivo mais recente
def df_tem_acerto2(df):
    """True só se o ano teve 2º turno (ACERTO2 com 0/1, não vazio/NaN)."""
    if "ACERTO2" not in df.columns:
        return False
    return df["ACERTO2"].fillna("").astype(str).str.strip().ne("").any()


def referencia_nomes_acertos(arquivos):
    for arq in reversed(arquivos):
        df = pd.read_csv(arq, sep=SEPARADOR, encoding=ENCODING, dtype=str)
        if {"SG_UF", "NM_MUNICIPIO"}.issubset(df.columns):
            return referencia_municipios(df)
    return None


def carregar_acertos_existentes(pasta=PASTA):
    arquivos = sorted(
        pasta.glob("acertos_mun_*.csv"),
        key=lambda p: int(p.stem.split("_")[-1]),
    )
    if not arquivos:
        raise FileNotFoundError(f"Nenhum acertos_mun_*.csv em {pasta.resolve()}")

    combinado = None
    anos = []

    for arq in arquivos:
        ano = int(arq.stem.split("_")[-1])
        anos.append(ano)
        df = pd.read_csv(arq, sep=SEPARADOR, encoding=ENCODING, dtype=str)
        df[CHAVE_CRUZAMENTO] = normalizar_cd_municipio(df[CHAVE_CRUZAMENTO])
        df = df.rename(columns={"ACERTO1": f"ACERTO1_{ano}"})
        df[f"ACERTO1_{ano}"] = df[f"ACERTO1_{ano}"].astype(int)
        cols_ano = [CHAVE_CRUZAMENTO, f"ACERTO1_{ano}"]

        if "ACERTO1b" in df.columns:
            df = df.rename(columns={"ACERTO1b": f"ACERTO1b_{ano}"})
            df[f"ACERTO1b_{ano}"] = df[f"ACERTO1b_{ano}"].astype(int)
            cols_ano.append(f"ACERTO1b_{ano}")
        else:
            print(f"  {ano}: sem ACERTO1b — regenere acertos_mun_{ano}.csv no passo 3")

        if df_tem_acerto2(df):
            df = df.rename(columns={"ACERTO2": f"ACERTO2_{ano}"})
            df[f"ACERTO2_{ano}"] = pd.to_numeric(df[f"ACERTO2_{ano}"], errors="coerce").astype(int)
            cols_ano.append(f"ACERTO2_{ano}")
        else:
            print(f"  {ano}: só 1º turno — ACERTO2 não entra no cruzamento")

        df = df[cols_ano]
        combinado = df if combinado is None else combinado.merge(df, on=CHAVE_CRUZAMENTO, how="inner")

    ref = referencia_nomes_acertos(arquivos)
    if ref is not None:
        combinado = ref.merge(combinado, on=CHAVE_CRUZAMENTO, how="right")

    colunas_acerto = [c for c in combinado.columns if c.startswith("ACERTO")]
    combinado["acertou_tudo"] = combinado[colunas_acerto].eq(1).all(axis=1)
    combinado["qtd_acertos"] = combinado[colunas_acerto].sum(axis=1)
    combinado["qtd_possivel"] = len(colunas_acerto)
    import re

    n_t1 = len([c for c in colunas_acerto if re.fullmatch(r"ACERTO1_\d{4}", c)])
    n_t1b = len([c for c in colunas_acerto if re.fullmatch(r"ACERTO1b_\d{4}", c)])
    n_t2 = len([c for c in colunas_acerto if re.fullmatch(r"ACERTO2_\d{4}", c)])
    print(
        f"Critérios de acerto: {len(colunas_acerto)} "
        f"({n_t1}× 1º colocado, {n_t1b}× 2º colocado no 1º turno, {n_t2}× 2º turno)"
    )

    return combinado, anos, arquivos


acertos_todos, anos_disponiveis, arquivos_acertos = carregar_acertos_existentes()

print("Arquivos encontrados:")
for arq in arquivos_acertos:
    print(f"  - {arq.name}")
print(f"\nAnos: {anos_disponiveis}")
print(f"Colunas de acerto: {[c for c in acertos_todos.columns if c.startswith('ACERTO')]}")

perfeitas = acertos_todos[acertos_todos["acertou_tudo"]].copy()
print(f"\nMunicípios que acertaram tudo ({len(perfeitas):,}):")
if perfeitas.empty:
    print("  Nenhum até agora — vale gerar o próximo ano (ex.: 2014).")
else:
    cols_exibir = ["SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO"] + [
        c for c in perfeitas.columns if c.startswith("ACERTO")
    ] + ["qtd_acertos"]
    cols_exibir = [c for c in cols_exibir if c in perfeitas.columns]
    display(perfeitas.sort_values(["SG_UF", "NM_MUNICIPIO"])[cols_exibir])

    ARQUIVO_PERFEITAS = PASTA / "municipios_acertaram_tudo.csv"
    COLUNAS_LISTA = ["SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO"]
    lista_perfeitas = (
        perfeitas[COLUNAS_LISTA]
        .drop_duplicates(subset=["CD_MUNICIPIO"])
        .sort_values(["SG_UF", "NM_MUNICIPIO"])
    )
    lista_perfeitas.to_csv(
        ARQUIVO_PERFEITAS, sep=SEPARADOR, encoding=ENCODING, index=False
    )
    print(f"\nLista salva: {ARQUIVO_PERFEITAS.resolve()} ({len(lista_perfeitas):,} municípios)")

# resumo: quase perfeitas (faltou 1 critério)
# quase = acertos_todos[acertos_todos["qtd_acertos"] == acertos_todos["qtd_possivel"] - 1]
# print(f"\nMunicípios com 1 erro apenas: {len(quase):,}")
# display(quase.head(10))

  1994: só 1º turno — ACERTO2 não entra no cruzamento
  1998: só 1º turno — ACERTO2 não entra no cruzamento
Critérios de acerto: 22 (8× 1º colocado, 8× 2º colocado no 1º turno, 6× 2º turno)
Arquivos encontrados:
  - acertos_mun_1994.csv
  - acertos_mun_1998.csv
  - acertos_mun_2002.csv
  - acertos_mun_2006.csv
  - acertos_mun_2010.csv
  - acertos_mun_2014.csv
  - acertos_mun_2018.csv
  - acertos_mun_2022.csv

Anos: [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]
Colunas de acerto: ['ACERTO1_1994', 'ACERTO1b_1994', 'ACERTO1_1998', 'ACERTO1b_1998', 'ACERTO1_2002', 'ACERTO1b_2002', 'ACERTO2_2002', 'ACERTO1_2006', 'ACERTO1b_2006', 'ACERTO2_2006', 'ACERTO1_2010', 'ACERTO1b_2010', 'ACERTO2_2010', 'ACERTO1_2014', 'ACERTO1b_2014', 'ACERTO2_2014', 'ACERTO1_2018', 'ACERTO1b_2018', 'ACERTO2_2018', 'ACERTO1_2022', 'ACERTO1b_2022', 'ACERTO2_2022']

Municípios que acertaram tudo (49):


,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,ACERTO1_1994,ACERTO1b_1994,ACERTO1_1998,ACERTO1b_1998,ACERTO1_2002,ACERTO1b_2002,ACERTO2_2002,...,ACERTO1_2014,ACERTO1b_2014,ACERTO2_2014,ACERTO1_2018,ACERTO1b_2018,ACERTO2_2018,ACERTO1_2022,ACERTO1b_2022,ACERTO2_2022,qtd_acertos
4732,GO,93041,PROFESSOR JAMIL,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,22
4920,GO,96237,TRÊS RANCHOS,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,22
1578,MG,40150,AGUANIL,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,22
1650,MG,40916,ASTOLFO DUTRA,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,22
1662,MG,41114,BARBACENA,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,22
1665,MG,41173,BARROSO,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,22
1671,MG,41270,BELO VALE,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,22
1684,MG,41491,BOM JARDIM DE MINAS,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,22
1804,MG,43451,CONCEIÇÃO DAS ALAGOAS,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,22
1823,MG,43770,CORDISBURGO,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,22



Lista salva: D:\cursos\python\estatisticas\tse\municipios_acertaram_tudo.csv (49 municípios)
